In [1]:
# %% Cell: THE BLENDER (Ensemble Logic)
import pandas as pd
import numpy as np
import os

# --- 1. SETUP PATHS ---
# Adjust these filenames to match EXACTLY what you have in your submissions folder
PATH_NN   = '../submissions/submission_champion_baseline.csv'  # Your Best NN
PATH_TREE = '../submissions/submission_titanium_v4_gauss.csv'          # Your Best XGB/LGBM

OUTPUT_PATH = '../submissions/submission_final_of_final.csv'

# --- 2. DEFINE WEIGHTS ---
# How much do you trust each model?
# 0.5 / 0.5 is standard.
# If NN is 0.8032 and Tree is 0.8100, give NN more weight (e.g., 0.6 / 0.4).
WEIGHT_NN   = 0.07
WEIGHT_TREE = 0.93

print(f"⚖️ BLENDING: {WEIGHT_NN} * NN  +  {WEIGHT_TREE} * TREE")

# --- 3. LOAD SUBMISSIONS ---
if not os.path.exists(PATH_NN) or not os.path.exists(PATH_TREE):
    print(f"❌ ERROR: One of the files is missing!")
    print(f"   Looking for: {PATH_NN}")
    print(f"   Looking for: {PATH_TREE}")
else:
    print("   📂 Loading submission files...")
    df_nn = pd.read_csv(PATH_NN)
    df_tree = pd.read_csv(PATH_TREE)

    # --- 4. SAFETY CHECK (Crucial!) ---
    # Ensure IDs match exactly. If row 1 is 'id_5' in NN and 'id_9' in Tree, blending fails.
    # We sort both to guarantee alignment.
    df_nn = df_nn.sort_values('id').reset_index(drop=True)
    df_tree = df_tree.sort_values('id').reset_index(drop=True)

    # Double check ids are identical
    if not df_nn['id'].equals(df_tree['id']):
        raise ValueError("❌ CRITICAL ERROR: IDs do not match between submissions!")
    
    print("   ✅ IDs aligned perfectly.")

    # --- 5. BLEND ---
    print("   🥣 Mixing predictions...")
    targets = ['target_short', 'target_medium', 'target_long']
    df_blend = pd.DataFrame()
    df_blend['id'] = df_nn['id']

    for col in targets:
        # The Magic Formula
        df_blend[col] = (df_nn[col] * WEIGHT_NN) + (df_tree[col] * WEIGHT_TREE)

    # --- 6. SAVE ---
    df_blend.to_csv(OUTPUT_PATH, index=False)
    print(f"🚀 SAVED ENSEMBLE: {OUTPUT_PATH}")
    
    # Preview
    print("\n   First 5 rows of blend:")
    print(df_blend.head())

⚖️ BLENDING: 0.07 * NN  +  0.93 * TREE
   📂 Loading submission files...
   ✅ IDs aligned perfectly.
   🥣 Mixing predictions...
🚀 SAVED ENSEMBLE: ../submissions/submission_final_of_final.csv

   First 5 rows of blend:
   id  target_short  target_medium  target_long
0   0      0.000243       0.002644     0.005530
1   1      0.000604       0.003379     0.006714
2   2      0.000447       0.003628     0.006918
3   3      0.000634       0.002628     0.007528
4   4      0.000362       0.002607     0.008814
